# B2.12 · Context engineering — cutting the false positives

**Function B — Application Security with an AI SDLC → The AI SDLC: an Agentic AppSec Pipeline, Before and After Deploy**  ·  *AI for Security*

Builds on **[B2.11 · Remediation engineering — proven in a sandbox before the merge request](https://spbreed.github.io/cyber-commons/lessons/B2.11.html)**.

| | |
|---|---|
| Tools used | tree-sitter, GLM-4.6, Llama 3.3, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Compare four context strategies against one bug and measure which are decidable and at what size.

**Why a security engineer needs it.** The model is given the repository and asked to be thorough, so the relevant line falls out of the window. The control it builds is: slice on the source-sink path, not on distance: the smallest context that still supports a severity decision.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Give an agent more context and it gets better, until it gets worse. The cliff is real, it arrives earlier than anyone expects, and past it you are paying more per token for a worse answer.

> **At CyberTravels.** Give the review agent CyberTravels' whole repository and it gets worse, not better. The cliff arrives earlier than anyone expects and you pay more per token for it.

## 2 · The framework

```
   accuracy
     ^
     |          .-----.
     |        .'       `.
     |      .'           `.       <- the cliff
     |    .'               `--....
     |  .'
     +--------------------------------> context tokens
        too little        enough      too much

   past the peak you pay more per token for a worse answer
```

Cross-cutting, and it applies to every stage that calls a model.

The instinct when a model gets something wrong is to give it more context. That
is usually backwards, and the reason is easiest to see if you stop counting
tokens and start counting **false positives**.

To decide whether code is vulnerable, a model needs exactly three things: the
**sink**, the **source**, and the **path** between them. Give it less than that
and the question is not hard — it is *unanswerable*, and a model asked an
unanswerable question does not say so. It answers from the only thing it has,
which is the shape of the line, and a confident verdict derived from the shape
of a line is precisely what a false positive is.

Give it more than that and nothing improves: a repository dumped into a prompt
produces a review of whatever survived truncation, and you cannot tell which
parts those were.

So context engineering is mostly **subtraction**, with one thing you must never
subtract: the **enclosing signature**, because that is where the source lives.
The identical concatenation is critical inside an HTTP handler and irrelevant
inside a migration script that takes a constant — and B2.5 has already shown you
the same line being a `sev 9` in one place and a deletion candidate in another.

B2.3 is the worked case, and it is worth reading as a context result rather than
a model result. Given a function, its signature and the caller's authority, the
model pass found a real missing-authorisation defect and quoted a line that
exists. Given a slice it could not decide, the same model returned CWE-89 at
0.71 confidence quoting a concatenation that is not in the file. Same model,
same prompt, different slice.

## 3 · The measurement, as a skill

One SQL injection, four candidate slices. The skill reports for each one whether the defect is **decidable** in it — sink, source and path all present — how much unrelated code it carries, and what the model does with it.

The ordering to watch: the smallest slice is not the best one, and the largest is not either.

### The skill — [`skills/appsec/context-window-sizing/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/appsec/context-window-sizing/SKILL.md)

```yaml
name: context-window-sizing
description: >-
  Find the smallest slice of a file in which a defect is actually decidable, and
  measure what larger windows add in unrelated code. Use when tuning what a
  model is shown per finding, when analysis costs too much, or when a model
  keeps missing a bug that is on the line you gave it.
allowed-tools: Read, Grep, Glob
```

# Decidable, then small — in that order

Context engineering for an analysis pipeline is not "send less". It is finding
the slice in which the question can be answered at all, and only then making it
smaller. A ±2-line window around the bug is cheap and **not decidable**: it
lacks the signature, so nothing in it says where the value came from.

## When to use this

When designing what a pipeline sends per finding, and whenever cost per finding
is the constraint.

## Procedure

**1 — Define decidability for the defect class.** For injection: the sink, the
value's origin, and any sanitiser between them. Write it down before slicing, or
you will judge slices by how they look.

**2 — Build the candidate slices.** The whole file, a fixed window around the
line, a wider window, and a **path slice** — the enclosing function plus the
definitions it depends on. Measure each in characters.

**3 — Mark each slice decidable or not,** against step 1. Cheap and undecidable
is the trap: it looks like a saving and it produces confident answers about
information that is not there.

**4 — Count unrelated content in the decidable ones.** Functions, constants and
imports the defect does not depend on. This is what a bigger window costs, in
tokens and in the model's attention.

**5 — Pick the smallest decidable slice, and say what it excluded.** The
exclusion list is what somebody re-reads when the pipeline misses something.

## Example

**Input** — the fixture committed at the top of [`scripts/context_window_sizing.py`](scripts/context_window_sizing.py). Edit it and re-run: the buckets, counts and verdicts below are derived from it, not hard-coded.

**Output** — the opening lines of a real run:

```
the bug is on line 22
whole file             839 chars   29 lines
±2 line window         207 chars    5 lines
±6 line window         432 chars   13 lines
strategy            sink   concat  source  intent  decidable
----------------------------------------------------------------
whole file          True   True    True    True    True
±2 line window      True   True    False   False   False
```

The run continues past this. The script is the example: `test_skills.py` executes it on every build, so this block cannot drift from what the skill actually prints.

## Output contract

```json
{
  "decidability": {"requires": ["str"]},
  "slices": [{"name": "str", "chars": 0, "decidable": false, "unrelated_units": 0}],
  "chosen": {"name": "str", "chars": 0, "unrelated_units": 0},
  "excluded": ["str"]
}
```

## Failure modes

- **Optimising size first.** An undecidable slice is not cheap; it is wrong at a
  lower price.
- **Judging slices by eye.** Write the decidability requirement down first.
- **Ignoring unrelated content** in a decidable slice. It is the cost you can
  actually remove.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/appsec/context-window-sizing/scripts/context_window_sizing.py
SCRIPT = "skills/appsec/context-window-sizing/scripts/context_window_sizing.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; sparse-checkout then materialises only the two directories a
    # lesson needs: the procedures, and the repository they are run against.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    # `skills` is the procedures; `cybertravels` is the sample repository they
    # scan. Both, or the scanning skills clone successfully and then find
    # nothing to look at.
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set",
                    "skills", "cybertravels"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## 4 · The rule, in one line

**Find the smallest slice in which the defect is decidable, and only then make
it smaller.**

Cutting below that line does not save money. It buys false positives at a
discount, and you pay for them again in triage.

## What you just proved

The whole file is roughly 840 characters, the ±2 window about 200 and the path slice about 390. The ±2 window is the cheapest and is not decidable — it has the sink and the concatenation but not the signature, so whether the value is user-controlled is unanswerable from it, and a model asked anyway answers from the shape of the line. The ±6 window and the whole file are decidable and carry unrelated functions. The path slice is the smallest decidable context, about 53% smaller than the whole file with zero unrelated functions.

## Your turn

Apply the path-slice rule where the source is three functions away from the sink. That is the case where text windows break down entirely and the call graph the threat model derives (B2.2) earns its keep — a ±N window can never contain a source that is in another file.

---

**Next → [B2.13 · Agentic AI in the pipeline — attesting control intent for agents and MCP servers](https://spbreed.github.io/cyber-commons/lessons/B2.13.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.12.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.12.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*